# N079 · 图的表示与建模

**目标：** 从关系、网格或依赖中明确顶点、边和方向。

**先修：** N018, N007, N053。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 有向/无向；邻接表/矩阵/边表；度；隐式图；自环与重边。

## 从问题到算法

图建模先明确顶点、边、方向和标签范围。邻接表适合稀疏图，必须保留孤立顶点；无向边要双向存储。邻接矩阵转无向边表只取上三角（含自环），否则每条边会重复。法官问题只需入度与出度，不必搜索图。

## 最小实现

**本章接口：** `build_adj_list(n, edges, directed)`、`adj_matrix_to_edges(matrix)`、`find_judge(n, trust)`

In [1]:
def build_adj_list(n,edges,directed=False):
    adj=[[] for _ in range(n)]
    for u,v in edges:
        if not (0<=u<n and 0<=v<n): raise ValueError('vertex outside 0..n-1')
        adj[u].append(v)
        if not directed and u!=v: adj[v].append(u)
    return adj

def adj_matrix_to_edges(matrix,directed=False):
    n=len(matrix)
    if any(len(row)!=n for row in matrix): raise ValueError('square matrix required')
    if not directed and any(bool(matrix[i][j])!=bool(matrix[j][i]) for i in range(n) for j in range(n)):
        raise ValueError('undirected adjacency matrix must be symmetric')
    return [(i,j) for i in range(n) for j in range(n) if matrix[i][j] and (directed or i<=j)]

def find_judge(n,trust):
    incoming=[0]*(n+1); outgoing=[0]*(n+1)
    for u,v in set(map(tuple,trust)): outgoing[u]+=1; incoming[v]+=1
    return next((i for i in range(1,n+1) if incoming[i]==n-1 and outgoing[i]==0),-1)

## 正确性、前提与复杂度

顶点集合独立于边集合创建，所以没有边的点也不会丢失。法官条件等价于“其他所有人都信任它，且它不信任任何人”，分别由入度n−1与出度0表达。

**代价：** 邻接表O(V+E)构建与存储；矩阵转换O(V²)扫描；法官O(n+E)平均时间。法官标签为1..n，与一般图0..n−1不同。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

edges=[(0,1),(1,2)]; adj=build_adj_list(4,edges,False)
show_table(['顶点','邻接顶点','度数'],[(i,a,len(a)) for i,a in enumerate(adj)])

顶点,邻接顶点,度数
0,[1],1
1,"[0, 2]",2
2,[1],1
3,[],0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert build_adj_list(4,[(0,1)],False)==[[1],[0],[],[]]
assert build_adj_list(3,[(0,1)],True)==[[1],[],[]]
assert adj_matrix_to_edges([[0,1,0],[1,0,1],[0,1,0]])==[(0,1),(1,2)]
assert find_judge(2,[[1,2]])==2
assert find_judge(1,[])==1
assert find_judge(3,[[1,3],[2,3],[3,1]])==-1
print('N079: 所有本章断言通过')

N079: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把棋盘移动规则建模为边。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 比较稀疏图与稠密图的存储成本。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 997. Find the Town Judge（warmup）
- 1971. Find if Path Exists in Graph（preview）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。